Cancer Cell Lines

source: https://singlecell.broadinstitute.org/single_cell/study/SCP542/pan-cancer-cell-line-heterogeneity

In [ ]:
%pip install aeacus scanpy anndata pandas numpy

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
from anndata import AnnData
from aeacus import Profiler
from aeacus.preprocessing import load_features

In [24]:
KINKER_DIR = Path("kinker_data/SCP542/")
UMI_PATH = KINKER_DIR / "other" / "UMIcount_data.txt"
METADATA_PATH = KINKER_DIR / "metadata" / "Metadata.txt"

In [25]:
#the model gene list
model_dir = Path(Profiler._default_pretrain_dir())
features = load_features(model_dir / "geneorder.tsv")
feature_to_idx = {gene: i for i, gene in enumerate(features)}

In [26]:
# keep model genes
with open(UMI_PATH) as f:
    cell_ids = f.readline().rstrip("\n").split("\t")[1:]

n_cells = len(cell_ids)
x = np.zeros((n_cells, len(features)), dtype=np.float32)
library_size = np.zeros(n_cells, dtype=np.float32)
found = set()

with open(UMI_PATH) as f:
    next(f)  # skip header
    for line in f:
        parts = line.rstrip("\n").split("\t")
        gene = parts[0]
        if gene in ("Cell_line", "Pool_ID"):
            continue
        values = np.asarray(parts[1: 1 + n_cells], dtype=np.float32)
        library_size += values

        if gene in feature_to_idx:
            x[:, feature_to_idx[gene]] = values
            found.add(gene)

In [27]:
missing = len(features) - len(found)
print(f"cells: {n_cells}, model genes found: {len(found)}/{len(features)}, missing: {missing} ({missing/len(features):.2%})")

cells: 56982, model genes found: 3735/3778, missing: 43 (1.14%)


In [28]:
# CPM + log1p (model argument can also be used)
library_size[library_size == 0] = 1.0
x = np.log1p((x / library_size[:, None]) * 1e6)

metadata = pd.read_csv(METADATA_PATH, sep="\t", skiprows=[1], index_col=0)
obs = metadata.reindex(cell_ids)
obs["true_label"] = 1

adata_kinker = AnnData(
    X=x,
    obs=obs,
    var=pd.DataFrame(index=features),
)

result_adata = (
    Profiler(
        test_input=adata_kinker,
        norm_type="already_normalized",
        use_raw=False,
        batch_size=8192,
    )
    .load()
    .profile()
)

print(result_adata.obs["malignancy_call"].value_counts())
print(f"Accuracy (sensitivity): {(result_adata.obs['malignancy_call'] == 'Malignant').mean():.4f}")

/tmp/ipykernel_2994057/3716219400.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(METADATA_PATH, sep="\t", skiprows=[1], index_col=0)


Model features: 3778
Missing features: 0 (0.00%)
malignancy_call
Malignant    56422
Normal         560
Name: count, dtype: int64
Accuracy (sensitivity): 0.9902
